Lucas Galindo - 202222210
Tomas Diaz - 202220658

Primero vamos a cargar los datos y tener una vista inicial de como estan organizados y medidas estadisticas generales para entender y buscar posibless anomalias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from importlib.metadata import version

df = pd.read_csv("Datos Lab 1.csv")
datos = df.copy()
datos.info()
datos.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639 entries, 0 to 1638
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient ID                    1639 non-null   object 
 1   Date of Service               1639 non-null   object 
 2   Sex                           1639 non-null   object 
 3   Age                           1571 non-null   float64
 4   Weight (kg)                   1566 non-null   float64
 5   Height (m)                    1578 non-null   float64
 6   BMI                           1586 non-null   float64
 7   Abdominal Circumference (cm)  1578 non-null   float64
 8   Blood Pressure (mmHg)         1639 non-null   object 
 9   Total Cholesterol (mg/dL)     1571 non-null   float64
 10  HDL (mg/dL)                   1557 non-null   float64
 11  Fasting Blood Sugar (mg/dL)   1585 non-null   float64
 12  Smoking Status                1639 non-null   object 
 13  Dia

,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Estimated LDL (mg/dL),CVD Risk Score
count,1571.000000,1566.000000,1578.000000,1586.000000,1578.000000,1571.000000,1557.000000,1585.000000,1571.000000,1563.000000,1578.000000,1554.000000,1582.000000,1610.000000
mean,46.803186,85.666006,1.757439,28.424744,91.538861,199.043673,56.183558,117.836860,175.770082,0.522440,125.632637,82.887536,113.235896,18.227281
std,13.039479,21.712504,0.118012,7.309275,13.427985,59.388670,16.721702,32.379634,11.695880,0.085692,22.577463,15.503625,61.435291,10.767666
min,6.134000,13.261000,1.371000,4.317000,49.542000,-1.256000,0.008000,15.306000,136.498000,0.250000,49.914000,31.720000,-92.055000,-20.057000
25%,37.000000,67.100000,1.666500,22.600000,79.700000,150.000000,42.000000,92.000000,167.000000,0.453000,108.000000,71.000000,62.000000,15.150000
50%,46.000000,86.314000,1.760000,28.000000,91.200000,199.000000,56.000000,115.000000,176.000000,0.519000,125.000000,82.000000,112.000000,16.967000
75%,55.000000,104.801500,1.850000,33.963000,102.267250,250.000000,70.000000,139.000000,185.000000,0.582000,141.000000,93.000000,159.000000,18.900000
max,89.420000,158.523000,2.146000,53.028000,136.336000,385.679000,110.315000,219.667000,214.394000,0.804000,202.711000,134.066000,317.314000,114.980000


In [ ]:
dict = pd.read_excel('DiccPacientes.xlsx')
pd.set_option('display.max_colwidth', None)
dict

,Nombre Columna,Tipo de dato,Comentarios
0,Patient ID,String,Identificador del paciente
1,Date of Service,Date,Fecha de la atención
2,Sex,String,"Sexo (Femenino, Masculino)"
3,Age,Integer,Edad
4,Weight (kg),Float,Peso
5,Height (m),Float,Altura
6,BMI,Float,Índice de masa corporal
7,Abdominal Circumference (cm),Float,Circunferencia abdominal
8,Blood Pressure (mmHg),String,"Presión sanguínea, de la forma ""<Presión arterial sistólica>/<Presión arterial diastólica>"""
9,Total Cholesterol (mg/dL),Float,Colesterol total


Viendo las estadisticas sabemos que en varias columnas hay valores nulos por lo que tendremos que reemplazar esos datos por la mediana de la respectiva columna que tenga el nulo. Ademas vemos un par de anomalias como mediciones negativas, datos irreales y una edad de 6 lo que no tendria sentido analizar ya que el estudio debe hacerse con adultos. Lo que vamos a solucionar.

In [ ]:
datos = datos[datos["Total Cholesterol (mg/dL)"] > 0]
datos = datos[datos["Estimated LDL (mg/dL)"] > 0]
datos = datos[datos["CVD Risk Score"] >= 0]
datos = datos[datos["BMI"] >= 15]
datos = datos[datos["Age"] >= 18]
datos = datos[datos["Systolic BP"] >= 70]
datos = datos[datos["Systolic BP"] <= 200]
datos = datos[datos["Diastolic BP"] >= 40]
datos = datos[datos["Diastolic BP"] <= 130]
datos = datos[datos["HDL (mg/dL)"] >= 20]
datos = datos[datos["Fasting Blood Sugar (mg/dL)"] >= 50]
datos = datos[datos["Fasting Blood Sugar (mg/dL)"] <= 250]
print(len(datos))

1095


Al quitar los datos anomalos ahora vamos a revisar si existen duplicados exactos y si hay varios registros de un paciente en diferentes fechas.

In [87]:
print(datos.duplicated().sum())
duplicados = datos.groupby(['Patient ID']).size()
duplicados = duplicados[duplicados > 1]
print(duplicados)

98
Patient ID
BQvQ6431    3
BqZp2317    2
CDsa2651    3
CKKa5109    3
DIVT3121    3
           ..
xbYu9929    3
yAsk5000    2
yvsn3005    2
zcgB3048    2
zxhX5525    2
Length: 114, dtype: int64


Vemos que hay varios duplicados que tenemos que borrar y registros de mismos paciente por lo que vamos a ver si es mejor dejar el ultimo registro o promediarlos o quitarlos.

In [88]:
datos.loc[datos.loc[:,'Patient ID']=="BQvQ6431"]

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Diabetes Status,Physical Activity Level,Family History of CVD,Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
130,BQvQ6431,09/11/2020,M,33.0,118.3,1.69,41.4,72.1,116/93,171.0,...,N,Moderate,N,0.427,116.0,93.0,Hypertension Stage 2,97.0,17.500,LOW
1469,BQvQ6431,09/11/2020,M,33.0,118.3,1.69,41.4,72.1,116/93,171.0,...,N,Moderate,N,0.427,116.0,93.0,Hypertension Stage 2,97.0,29.833,LOW
1544,BQvQ6431,09/11/2020,M,33.0,118.3,1.69,41.4,72.1,116/93,171.0,...,N,Moderate,N,0.427,116.0,93.0,Hypertension Stage 2,97.0,17.500,LOW


Al ver algunos de los registros que tienen mismo ID vemos que son los duplicados y no existen registros en diferentes fechas de un mismo paciente. Sin embargo hay grupos de 3 donde 2 son iguales y otro tiene un CVD Risk Score diferente. Decidimos borrar ambos registros en los casos que tienen diferente score ya que no tenemos la experticia en el tema para saber cual es el score correcto y si promediarlos pueda dar un valor erroneo para el entrenamiento. De igual manera tenemos una gran cantidad de datos y es mejor tener una muestra de buena calidad a una un poco mas grande pero con mayor incertidumebre. 

In [89]:
datos = datos.drop_duplicates()
datos = datos.groupby(['Patient ID', 'Date of Service']).filter(lambda x: x['CVD Risk Score'].nunique() <= 1)
duplicados = datos.groupby(['Patient ID']).size()
duplicados = duplicados[duplicados > 1]
print(duplicados)
print(datos.duplicated().sum())
print(len(datos))

Series([], dtype: int64)
0
853


Ahora que tenemos los datos limpios quitamos las columnas Id  y Date porque no son relevantes para el modelo.

In [ ]:
datos = datos.drop(columns=['Patient ID', 'Date of Service', 'Height (cm)'])